# Transformer Language Model Architecture

A language model takes as input a batched sequence of integer token IDs (i.e., `torch.Tensor` of shape
(`batch_size`, `sequence_length`)), and returns a (batched) normalized probability distribution over the
vocabulary (i.e., a PyTorch Tensor of shape (`batch_size`, `sequence_length`, `vocab_size`)), where the
predicted distribution is over the next token for each input token. 

When training the language model, we use these next-token predictions to calculate the cross-entropy loss between the actual next token and the predicted next token. When generating text from the language model during inference, we take the predicted next-token distribution from the final time step (i.e., the last item in the sequence) to generate the next token in the sequence (e.g., by taking the token with the highest probability, sampling from the distribution, etc.), add the generated token to the input sequence, and repeat.

In this project, we will build this Transformer language model from scratch.

## Parameter Initialization

Pre-norm transformers are unusually robust to initializations, but they can still have a significant impact on training speed and convergence.

For now, use these approximate initializations (Normal Distribution here refer to the classic Gaussian bell curve distribution):

(a) Linear weights (e.g., in Feed Forward Neural Nets): $N(\mu = 0, \sigma^2 = \frac{2}{d_{in}+d_{out}})$, truncated at $[-3 \sigma, 3 \sigma]$, where $d_{in}$ and $d_{out}$ refer to the input and output dimensions respectively.

(b) Embedding (in the LLM context): $N(\mu = 0, \sigma^2 = 1)$, truncated at $[-3, 3]$.

(c) RMSNorm (in the LLM context): $\mathbb{1}$ -- uniformly 1's.

You should use `torch.nn.init.trunc_normal_` to initialize the truncated normal weights.


## Linear and Embedding Modules, RoPE, RMS Normalization

### Linear Module

Following most modern LLMs, we will not include a bias term.

#### Coding task for Linear Module:

Implement a `Linear` Python class that inherits from `torch.nn.Module` and performs a linear transformation. Your implementation should follow the following interface. This is intended to resemble the interface of PyTorch’s built-in `nn.Linear` module, except for not having a bias argument or parameter.

- `def __init__(self, in_features, out_features, device=None, dtype=None)` 
  - Construct a linear transformation module. This function should accept the following parameters:
    - `in_features`: `int` 
      final dimension of the input
    - `out_features`: `int` 
      final dimension of the output
    - `device`: `torch.device | None = None` 
      Device to store the parameters on
    - `dtype`: `torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Apply the linear transformation to the input.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) construct and store your parameter as W, putting it in an `nn.Parameter`. NOTE: $W \in \mathbb{R}^{d_{out} \times d_{in}}$ is in **row-major** form, where each row corresponds to one output feature. In `forward`, compute the transformation as $x W^T$ (equivalently, `x @ W.T`) over the final input dimension, preserving any leading dimensions of `x`.

(iv) do **not** use `nn.Linear` or `nn.functional.linear`

For initializations, use the settings from above along with `torch.nn.init.trunc_normal_` to initialize the weights.

To test your `Linear` module, implement the test adapter at [adapters.run_linear]. The adapter should load the given weights into your `Linear` module. You can use `Module.load_state_dict` for this purpose. Then, run `uv run pytest -k test_linear` and check that all unit tests pass.

### Embedding Module

Given a sequence of token IDs, the Transformer language model uses an input embedding to convert token IDs to dense vectors, passes the embedded tokens through `num_layers` Transformer blocks, and then applies a learned linear projection (the “output embedding” or “LM head”) to produce the predicted next-token logits.

The first layer of the Transformer is an embedding layer that maps integer token IDs into a vector space of dimension `d_model`. 

We will implement a custom `Embedding` class that inherits from `torch.nn.Module` (so you should not use `nn.Embedding`). The forward method should select the embedding vector for each token ID by indexing into an embedding matrix of shape (`vocab_size`, `d_model`) using a `torch.LongTensor` of token IDs with shape (`batch_size`, `sequence_length`).

#### Coding task for Embedding Module

Implement the `Embedding` class that inherits from `torch.nn.Module` and performs an embedding lookup. Your implementation should use the following interface:

- `def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None)` 
  - Construct an embedding module. This function should accept the following parameters:
    - `num_embeddings`: `int` 
      Size of the vocabulary
    - `embedding_dim` : `int`
      Dimension of the embedding vectors, i.e., `d_model`
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, token_ids: torch.Tensor) -> torch.Tensor` 
  - Lookup the embedding vectors for the given token IDs.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) initialize your embedding matrix as an `nn.Parameter`

(iv) store the embedding matrix with the `d_model` being the final dimension

(v) do **not** use `nn.Embedding` or `nn.functional.embedding`

Again, use the settings from above for initialization, and use `torch.nn.init.trunc_normal_` to initialize the weights. 

To test your implementation, implement the test adapter at [adapters.run_embedding]. Then, run `uv run pytest -k test_embedding` and check that all tests pass.

### RoPE (Rotary Positional Embeddings)

To inject positional information into the model, we will implement Rotary Position Embeddings (RoPE). For a given query token $q^{(i)} = W_q x^{(i)} \in \mathbb{R}^d$ at token position $i$, we will apply a pairwise rotation matrix $R^i$, giving us $(q')^{(i)} = R^i q^{(i)} = R^i W_q x^{(i)}$. Here, $R^i$ will rotate pairs of embedding elements $q^{(i)}_{2k-1:2k}$ as $2d$-vectors by the angle $\theta_{i,k} = \frac{i}{\Theta^{(2k-2)/d}}$ for $k \in \{1, \ldots, d/2\}$ and some constant $\Theta$. Thus, we can consider $R^i$ to be a block-diagonal matrix of size $d \times d$, with blocks $R^i_k$ for $k \in \{1, \ldots, d/2\}$, with

$R^i_k =
\begin{pmatrix}
\cos(\theta_{i,k}) & -\sin(\theta_{i,k}) \\
\sin(\theta_{i,k}) & \cos(\theta_{i,k})
\end{pmatrix}$

Thus we get the full rotation matrix

$R^i =
\begin{pmatrix}
R^i_1 & 0 & 0 & \cdots & 0 \\
0 & R^i_2 & 0 & \cdots & 0 \\
0 & 0 & R^i_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & \cdots & R^i_{d/2}
\end{pmatrix}$,

where the $0$'s represent $2 \times 2$ zero matrices. While one could construct the full $d \times d$ matrix, a good solution should use the properties of this matrix to implement the transformation more efficiently. Since we only care about the relative rotation of tokens within a given sequence, we can reuse the values we compute for $\cos(\theta_{i,k})$ and $\sin(\theta_{i,k})$ across layers, and different batches. If you would like to optimize it, you may use a single RoPE module referenced by all layers, and it can have a $2d$ pre-computed buffer of $\sin$ and $\cos$ values created during `init` with `self.register\_buffer(persistent=False)`, instead of an `nn.Parameter` because we do not want to learn these fixed cosine and sine values. The exact same rotation process we did for our $q^{(i)}$ is then done for $k^{(j)}$, rotating by the corresponding $R^j$. Notice that this layer has no learnable parameters.

#### Coding Task for RoPE

Implement a Python class `RotaryPositionalEmbedding` that applies RoPE to the input tensor. The following interface is to be used:

- `def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None)` 
  - Construct the RoPE module and create buffers if needed.
    - `theta`: `float` 
      $\Theta$ value for the RoPE
    - `d_k`: `int` 
      dimension of query and key vectors
    - `max_seq_len`: `int` 
      Maximum sequence length that will be input
    - `device: torch.device | None = None` 
      Device to store the buffer on

- `def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor` 
  - Process an input tensor of shape (..., `seq_len`, `d_k`) and return a tensor of the same shape. Note that you should tolerate $x$ with an arbitrary number of batch dimensions. You should assume that the token positions are a tensor of shape (..., `seq_len`) specifying the token positions of $x$ along the sequence dimension.

You should use the token positions to slice your (possibly precomputed) $\cos$ and $\sin$ tensors along the sequence dimension. To test your implementation, complete [adapters.run_rope] and make sure it passes `uv run pytest -k test_rope`.

### Root Mean Square Layer Normalization

We will use root mean square layer normalization.

Given a vector $a \in \mathbb{R}^{d_{model}}$ of activations, `RMSNorm` will scale each activation $a_i$ according to the formula

$\operatorname{RMSNorm}(a_i) = \frac{a_i}{\operatorname{RMS(a)}} \times g_i$,

where $\operatorname{RMS}(a) = \sqrt{ \frac{1}{d_{model}} \times ( \sum_{i=1}^{d_{model}} (a_i)^2 ) + \epsilon }$. 

Here, $g_i$ is a learnable parameter (there are `d_model` such parameters in total), and $\epsilon$ is a hyperparameter often fixed at +1e-05. 

You should upcast your input to `torch.float32` to prevent overflow when you square the input. Overall, your `forward` method should look like:

```python
in_dtype = x.dtype
x = x.to(torch.float32)
# Your code here performing RMSNorm
# ...
result = computed_result
# Return the result in the original dtype
return result.to(in_dtype)
```

#### Coding tasks for RMS Normalization

Implement `RMSNorm` as a `torch.nn.Module`. This Python class should use the following interface:

- `def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None)` 
  - Construct the RMSNorm module. This function should accept the following parameters:
    - `d_model`: `int` 
      Hidden dimension of the model
    - `eps: float = 1e-5` 
      Epsilon value for numerical stability
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Process an input tensor of shape (`batch_size`, `sequence_length`, `d_model`) and return a tensor of the same shape.

Note: Remember to upcast your input to `torch.float32` before performing the normalization (and later downcast to the original `dtype`), as described above.
To test your implementation, implement the test adapter at [adapters.run_rmsnorm]. Then, run `uv run pytest -k test_rmsnorm` and make sure all tests pass.

In [ ]:
from __future__ import annotations

import math

import torch


class Linear(torch.nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.nn.Parameter(torch.empty((out_features, in_features), device=device, dtype=dtype))

        std = math.sqrt(2.0 / (in_features + out_features))
        torch.nn.init.trunc_normal_(self.W, mean=0.0, std=std, a=-3.0 * std, b=3.0 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.W.T


class Embedding(torch.nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = torch.nn.Parameter(torch.empty((num_embeddings, embedding_dim), device=device, dtype=dtype))

        torch.nn.init.trunc_normal_(self.weight, mean=0.0, std=1.0, a=-3.0, b=3.0)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]


class RotaryPositionalEmbedding(torch.nn.Module):
    def __init__(
        self,
        theta: float,
        d_k: int,
        max_seq_len: int,
        device: torch.device | None = None,
    ) -> None:
        super().__init__()
        if d_k % 2 != 0:
            raise ValueError("d_k must be even for RoPE pairwise rotations")

        self.theta = theta
        self.d_k = d_k
        self.max_seq_len = max_seq_len

        positions = torch.arange(max_seq_len, device=device, dtype=torch.float32)
        dimension_indices = torch.arange(0, d_k, 2, device=device, dtype=torch.float32)
        angles = positions[:, None] / (theta ** (dimension_indices / d_k))
        self.register_buffer("cos", torch.cos(angles), persistent=False)
        self.register_buffer("sin", torch.sin(angles), persistent=False)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        token_positions = token_positions.to(device=self.cos.device, dtype=torch.long)
        cos = self.cos[token_positions].to(device=x.device)
        sin = self.sin[token_positions].to(device=x.device)

        while cos.ndim < x.ndim:
            cos = cos.unsqueeze(-3)
            sin = sin.unsqueeze(-3)

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos
        result = torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)
        return result.to(in_dtype)


class RMSNorm(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        eps: float = 1e-5,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones((d_model,), device=device, dtype=dtype))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        result = x / rms * self.weight.to(torch.float32)
        return result.to(in_dtype)


### How the `Linear`, `Embedding`, `RotaryPositionalEmbedding`, and `RMSNorm` classes work

All four classes inherit from `torch.nn.Module`, so PyTorch tracks their parameters and buffers through the module machinery. Calling `super().__init__()` sets up that machinery before any parameters or buffers are assigned. Each learnable tensor is wrapped in `torch.nn.Parameter`; assigning a `Parameter` to an attribute registers it in the module state dict, includes it in `module.parameters()`, and allows autograd to accumulate gradients into it during backpropagation.

`Linear` stores one parameter, `W`, whose shape is `(out_features, in_features)`. This is a row-major layout for linear-layer weights: each row contains the weights for one output coordinate, and each column corresponds to one input coordinate. If the input tensor `x` has shape `(..., in_features)`, the leading dimensions `...` can be a batch, a sequence, or any other collection of positions. The expression `self.W.T` views the weight matrix with shape `(in_features, out_features)`, and `x @ self.W.T` performs matrix multiplication over only the final input dimension. The result therefore has shape `(..., out_features)`: all leading dimensions are preserved, and the last dimension is replaced by the output-feature dimension. There is no bias tensor, matching the assignment specification.

The `Linear` initializer creates uninitialized storage with `torch.empty`, wraps it as `W`, then fills it in place with `torch.nn.init.trunc_normal_`. The standard deviation is `sqrt(2 / (in_features + out_features))`, so the initialization scale depends on both the fan-in and fan-out of the layer. The lower and upper truncation bounds are `-3 * std` and `3 * std`, which keeps sampled weights within three standard deviations of the zero mean.

`Embedding` stores one parameter, `weight`, whose shape is `(num_embeddings, embedding_dim)`. The first axis is a lookup table over vocabulary IDs: row `i` is the vector assigned to token ID `i`. The final axis is the dense embedding dimension `d_model`, so selecting rows naturally appends that vector dimension to the input token-ID shape. If `token_ids` has shape `(... )`, direct indexing with `self.weight[token_ids]` returns a tensor of shape `(..., embedding_dim)`. For example, a token-ID tensor of shape `(batch_size, sequence_length)` becomes embedded token vectors of shape `(batch_size, sequence_length, embedding_dim)`.

The `Embedding` initializer also uses `torch.empty` followed by `torch.nn.init.trunc_normal_`, but with mean `0`, standard deviation `1`, and fixed bounds `[-3, 3]`. The forward pass does not perform matrix multiplication. It uses tensor indexing to gather rows from the embedding table, so each integer token ID is replaced by the corresponding learned vector while the original token-ID layout is preserved as the leading dimensions of the output tensor.

`RotaryPositionalEmbedding` has no learnable parameters. During initialization, it builds the fixed RoPE angles for every cached position from `0` through `max_seq_len - 1` and for every even coordinate pair in the `d_k` query/key dimension. It stores `cos` and `sin` with `register_buffer(..., persistent=False)`, which means the tensors move with the module across devices but are not trained and are not saved as learned model weights.

The RoPE forward pass accepts `x` with shape `(..., sequence_length, d_k)` and `token_positions` with shape `(..., sequence_length)`. It indexes the cached cosine and sine buffers at those token positions, adds singleton dimensions before the sequence axis when needed so the position tensors broadcast across extra batch or head axes, and then splits the final feature dimension into even and odd coordinates. Each pair is rotated by computing `x_even * cos - x_odd * sin` for the even coordinate and `x_even * sin + x_odd * cos` for the odd coordinate. Finally, `torch.stack(..., dim=-1).flatten(-2)` interleaves the rotated pairs back into the original final dimension, and the result is returned in the input dtype.

`RMSNorm` stores one learnable scale parameter, `weight`, whose shape is `(d_model,)`. This corresponds to the vector of $g_i$ values in the RMSNorm formula, one scale value for each coordinate in the final activation dimension. The initializer uses `torch.ones` so the normalization starts as pure root-mean-square rescaling with no learned coordinate-specific change. The module also stores `d_model` and `eps` for clarity and for the normalization denominator.

The `RMSNorm` forward pass first records the input dtype and converts the activations to `torch.float32`. This keeps the square-and-average computation more stable for lower-precision inputs. It then computes `x * x`, averages over the final dimension with `keepdim=True`, adds `eps`, and takes the square root to get a denominator that broadcasts across the original input shape. Dividing `x` by this denominator normalizes each vector over its last dimension, multiplying by `weight` applies the learned per-coordinate scale, and `result.to(in_dtype)` returns the output in the same dtype as the input.


## Position-Wise Feed-Forward Network

The SiLU or Swish activation function is defined as follows:

$\operatorname{SiLU}(x) = x \times \sigma(x) = \frac{x}{1+e^{-x}}$.

Here, $\sigma(x)$ is the (logistic) sigmoid function.

The SiLU function is similar to the ReLU function (rectified linear unit) but is smooth at 0. 

Gated Linear Units (GLUs) are defined as the **element-wise product** of one linear transformation and a sigmoid-gated linear transformation:

$\operatorname{GLU}(x,W_1,W_2) = (xW_2^T) \odot \sigma(xW_1^T)$,

where $\odot$ represents element-wise multiplication, also known as the Hadamard product (to be distinguished from matrix multiplication).

Under the assumption that biases are omitted, the SwiGLU feed-forward layer, a GLU variant that uses a SiLU/Swish gate, can be expressed as follows:

$\operatorname{SwiGLU}(x,W_1,W_2,W_3) = (\operatorname{SiLU}(xW_1^T) \odot xW_3^T)W_2^T$,

where $x \in \mathbb{R}^{d_{model}}$, $W_1,W_3 \in \mathbb{R}^{d_{ff} \times d_{model}}$, $W_2 \in \mathbb{R}^{d_{model} \times d_{ff}}$, and $d_{ff} = \frac{8}{3} \times d_{model}$ (rounded in practice) -- this definition of $d_{ff}$ is common in SwiGLU Transformer feed-forward networks because it keeps the parameter count roughly comparable to a standard feed-forward network with hidden width $4 \times d_{model}$.

We shall use SwiGLU for our feed forward network:

$\operatorname{FFN}(x) = \operatorname{SwiGLU}(x,W_1,W_2,W_3)$.

### Coding task for position-wise feed-forward network

Implement the SwiGLU feed-forward network.

Note: Manually implement a numerically stable sigmoid from elementary functions. You should also set $d_{ff}$ to $\frac{8}{3} \times d_{model}$ rounded to the nearest multiple of 64 in your implementation. Accept $d_{ff}$ explicitly when provided by tests/configs, and only compute using the formula described if no $d_{ff}$ is supplied. Assume that:

- Input shape is (..., `d_model`).

- $W_1$ and $W_3$ are up-projections of shape (`d_ff`, `d_model`).

- $W_2$ is the down-projection of shape (`d_model`, `d_ff`).

To test your implementation against our provided tests, you will need to implement the test adapter at [adapters.run_swiglu]. Then, run `uv run pytest -k test_swiglu` to test your implementation.

In [ ]:
import torch

from cs336_basics.nn_linear_embedding_rope_rmsnorm import Linear


def swiglu_d_ff(d_model: int) -> int:
    raw_d_ff = 8.0 * d_model / 3.0
    return max(64, int((raw_d_ff + 32.0) // 64.0) * 64)


def stable_sigmoid(x: torch.Tensor) -> torch.Tensor:
    z = torch.exp(-torch.abs(x))
    return torch.where(x >= 0, 1.0 / (1.0 + z), z / (1.0 + z))


def silu(x: torch.Tensor) -> torch.Tensor:
    return x * stable_sigmoid(x)


class SwiGLU(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int | None = None,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff if d_ff is not None else swiglu_d_ff(d_model)
        self.w1 = Linear(d_model, self.d_ff, device=device, dtype=dtype)
        self.w2 = Linear(self.d_ff, d_model, device=device, dtype=dtype)
        self.w3 = Linear(d_model, self.d_ff, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(silu(self.w1(x)) * self.w3(x))
